# Phase E.1.h5 — Hybrid KAN at Akimbo v1.0 architecture

First real Phase E run. Hybrid topology: keep the (768×4hm → 1024) FT with factoriser + 4 input buckets + horizontal king mirror from Akimbo v1.0, but replace the entire v1.0 output stack (`screlu(16) → screlu(32) → Linear(1)`) with a single `ReluKAN(2048, 1) G=5 k=3` layer. Tests Phase C's hypothesis ("hybrid > full") and Q1 ("does ReLU-KAN's bell-shape advantage survive at the wider architecture") at the same time.

**Budget**: 800 superbatches × ~34 s/SB (from E.0 smoke) ≈ **7.6 h on A100, ~$9 on Colab Pro+**.

**Checkpointing**: `save_rate=50` → 16 checkpoints per run. Per the Phase E plan: pick the winning checkpoint by SPRT, never by training loss.

**Data**: single January 2024 binpack for the first run (matches E.0 setup). Multi-month staging is a follow-up if h5 looks promising.

**Runtime**: Colab Pro+ with A100. Plan to keep the tab open for ~8 h. If the kernel disconnects mid-run, the local `/content/bullet/checkpoints/` keeps the last completed checkpoint — you can re-mount Drive and recover.

## 1. Install Rust + clone repo

In [ ]:
%%bash
if ! command -v cargo &> /dev/null; then
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    echo 'source $HOME/.cargo/env' >> ~/.bashrc
fi
source $HOME/.cargo/env
rustc --version
cargo --version

In [ ]:
%%bash
set -e
rm -rf /content/bullet
cd /content
git clone https://github.com/y0sif/bullet.git
cd bullet
git log -1 --oneline
echo '---'
ls examples/phase_e1_h5.rs
grep -A1 'phase_e1_h5' crates/bullet_lib/Cargo.toml || echo 'WARN: phase_e1_h5 not in Cargo.toml'

## 2. Confirm A100 is attached

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 3. Download January 2024 binpack (~7.7 GB compressed, ~25 GB decompressed)

In [ ]:
%%bash
apt-get install -y zstd 2>/dev/null || true

mkdir -p /content/bullet/data
cd /content/bullet/data

if [ ! -f test80-2024-01-jan.binpack ]; then
    echo "Downloading test80-2024 January shard (~7.7 GB compressed)..."
    wget --progress=dot:giga -O test80-2024-01-jan.binpack.zst \
        "https://huggingface.co/datasets/linrock/test80-2024/resolve/main/test80-2024-01-jan-2tb7p.min-v2.v6.binpack.zst"
    echo "Decompressing..."
    zstd -d test80-2024-01-jan.binpack.zst -o test80-2024-01-jan.binpack --rm
fi

ls -lh test80-2024-01-jan.binpack
df -h /content

## 4. Build the E.1.h5 binary

In [ ]:
%%bash
source $HOME/.cargo/env
cd /content/bullet
cargo build --release --example phase_e1_h5 2>&1 | tail -20

## 5. Run training (~7.6 h)

Streams output to a log file. The cell will block until training finishes; the kernel must stay alive. Bullet saves a checkpoint every 50 superbatches into `/content/bullet/checkpoints/phase_e1_h5-<sb>/`.

In [ ]:
import subprocess, sys, os, shutil, time

os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

log_path = "/content/phase_e1_h5_log.txt"
ckpt_dir = "/content/bullet/checkpoints"
if os.path.isdir(ckpt_dir):
    shutil.rmtree(ckpt_dir, ignore_errors=True)

print(f"Logging to {log_path}")
start = time.time()

proc = subprocess.Popen(
    ["cargo", "run", "--release", "--example", "phase_e1_h5"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    cwd="/content/bullet", text=True, bufsize=1,
)
with open(log_path, "w") as log:
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        log.write(line)
proc.wait()

elapsed = time.time() - start
print(f"\nExit code: {proc.returncode}")
print(f"Wall-clock: {elapsed/3600:.2f} h ({elapsed:.0f} s)")
if proc.returncode != 0:
    print("WARNING: training exited non-zero. Inspect log; checkpoints on disk may still be usable.")

## 6. Parse loss + throughput, plot loss curve

In [ ]:
import re
import matplotlib.pyplot as plt

def strip_ansi(s):
    return re.sub(r'\x1b\[[0-9;]*m', '', s)

log_path = "/content/phase_e1_h5_log.txt"
loss_records = []
time_records = []

with open(log_path) as f:
    for line in f:
        line = strip_ansi(line)
        m_loss = re.search(r'superbatch\s+(\d+)\s+\|.*?running loss\s+([\d.]+)', line)
        if m_loss:
            loss_records.append((int(m_loss.group(1)), float(m_loss.group(2))))
        m_time = re.search(r'superbatch\s+(\d+)\s+\|.*?time\s+(\d+\.\d+)s', line)
        if m_time:
            time_records.append((int(m_time.group(1)), float(m_time.group(2))))

if not loss_records:
    print("WARNING: no loss lines parsed.")
else:
    first_sb, first_loss = loss_records[0]
    last_sb, last_loss = loss_records[-1]
    print(f"Loss: SB {first_sb}  {first_loss:.6f}  ->  SB {last_sb}  {last_loss:.6f}")
    print(f"Relative drop: {(first_loss - last_loss) / first_loss * 100:.1f}%")
    print(f"Completed {last_sb}/800 superbatches")

if time_records:
    secs = [t for _, t in time_records]
    mean_sb = sum(secs) / len(secs)
    print(f"\nMean per-superbatch wall-clock: {mean_sb:.1f} s")
    print(f"Total training time: {mean_sb * len(secs) / 3600:.2f} h")

if loss_records:
    fig, ax = plt.subplots(figsize=(12, 5))
    xs = [sb for sb, _ in loss_records]
    ys = [l for _, l in loss_records]
    ax.plot(xs, ys, linewidth=1.5)
    ax.set_xlabel("Superbatch")
    ax.set_ylabel("Running loss")
    ax.set_title("Phase E.1.h5 — Hybrid ReLU-KAN at v1.0 architecture")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("/content/phase_e1_h5_loss.png", dpi=150)
    plt.show()

## 7. Save log + plot + all 16 checkpoints to Drive

Each checkpoint will become a build target on the engine side. Per the Phase E plan: build an Akimbo binary from each saved checkpoint and SPRT to pick the winner — never use training loss to rank checkpoints.

In [ ]:
import shutil, os, glob
from google.colab import drive
drive.mount('/content/drive')

dest = '/content/drive/MyDrive/kanue/phase_e1_h5'
os.makedirs(dest, exist_ok=True)

for src in ['/content/phase_e1_h5_log.txt', '/content/phase_e1_h5_loss.png']:
    if os.path.exists(src):
        shutil.copy(src, dest)

for ckpt in sorted(glob.glob('/content/bullet/checkpoints/phase_e1_h5-*')):
    name = os.path.basename(ckpt)
    out_dir = os.path.join(dest, name)
    os.makedirs(out_dir, exist_ok=True)
    for fname in ['quantised.bin', 'raw.bin', 'optim']:
        src = os.path.join(ckpt, fname)
        if os.path.exists(src) and os.path.isfile(src):
            shutil.copy(src, out_dir)
    print(f'Saved {name}')

print(f'\nDestination: {dest}')